# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# .env sits at the repo root (two folders up from here) and holds:
#
#   OPENAI_API_KEY   - used for both chat and embeddings
#   OPENAI_BASE_URL  - the course gateway. The OpenAI SDK reads this straight
#                      from the environment, so Chroma's embedding function
#                      inherits it without being told.
#   TAVILY_API_KEY   - web search. Not used in this notebook; needed in part 02.
#
# The starter also asked for CHROMA_OPENAI_API_KEY. It is not needed on
# chromadb 1.5.9 -- OpenAIEmbeddingFunction switches to OPENAI_API_KEY whenever
# that one is set. Confirmed by the preflight below, which reports it MISSING
# while the embedding check still passes.

In [5]:
# Load environment variables from the .env file at the repo root.
# override=True so edits to .env win over anything already set in the shell.
load_dotenv(dotenv_path="../../.env", override=True)

True

In [6]:
# Preflight: prove every external service works before building anything on top of them.
# Reports all failures instead of stopping at the first one.

from openai import OpenAI
from tavily import TavilyClient

results = []

def check(label, fn):
    """Run one check, record pass/fail, never raise."""
    try:
        results.append((label, "PASS", fn()))
    except Exception as e:
        results.append((label, "FAIL", f"{type(e).__name__}: {e}"))

# 1. Which keys did .env actually give us? Print presence only, never values.
for key in ["OPENAI_API_KEY", "OPENAI_BASE_URL", "CHROMA_OPENAI_API_KEY", "TAVILY_API_KEY"]:
    val = os.getenv(key)
    results.append((f"env: {key}", "PASS" if val else "MISSING", f"{len(val)} chars" if val else "-"))

# 2. Chat endpoint. Uses OPENAI_BASE_URL automatically if it is set.
def chat_check():
    client = OpenAI()
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Reply with the single word: ok"}],
        max_tokens=5,
    )
    return r.choices[0].message.content.strip()

# 3. Embeddings via the OpenAI SDK. Same client, same key as chat.
def embed_sdk_check():
    client = OpenAI()
    r = client.embeddings.create(model="text-embedding-3-small", input="Gran Turismo")
    return f"{len(r.data[0].embedding)} dims"

# 4. Embeddings via Chroma's own function -- this is the one that matters.
#    Chroma builds a SEPARATE client, so a pass here proves nothing about check 3 and vice versa.
def embed_chroma_check():
    fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="text-embedding-3-small",
    )
    return f"{len(fn(['Gran Turismo'])[0])} dims"

# 5. Web search.
def tavily_check():
    r = TavilyClient(api_key=os.getenv("TAVILY_API_KEY")).search("Mortal Kombat X PlayStation 5", max_results=1)
    return f"{len(r.get('results', []))} result(s)"

check("openai: chat", chat_check)
check("openai: embeddings (SDK)", embed_sdk_check)
check("chroma: embedding function", embed_chroma_check)
check("tavily: search", tavily_check)

results.append(("chromadb version", "INFO", chromadb.__version__))

width = max(len(r[0]) for r in results)
for label, status, detail in results:
    print(f"{label:<{width}}  {status:<8} {detail}")


env: OPENAI_API_KEY         PASS     49 chars
env: OPENAI_BASE_URL        PASS     30 chars
env: CHROMA_OPENAI_API_KEY  MISSING  -
env: TAVILY_API_KEY         PASS     57 chars
openai: chat                PASS     ok
openai: embeddings (SDK)    PASS     1536 dims
chroma: embedding function  PASS     1536 dims
tavily: search              PASS     1 result(s)
chromadb version            INFO     1.5.9


### VectorDB Instance

In [7]:
# PersistentClient writes to disk, so the collection survives a kernel restart.
# chromadb.Client() would keep everything in memory and lose it on restart.
# The path is relative to the folder Jupyter was started in -> project/starter/chromadb/
chroma_client = chromadb.PersistentClient(path="chromadb")

print("client ready ->", chroma_client)

client ready -> <chromadb.api.client.Client object at 0x000001CD06A4E6D0>


### Collection

In [8]:
# The embedding function turns text into numbers. It runs twice:
#   - once per document when we add games
#   - once per question when we search
# Both must use the SAME model, or the numbers are not comparable.

# model_name is passed explicitly on purpose: Chroma's default is the older
# text-embedding-ada-002. Both give 1536 numbers, so a wrong default would be
# invisible -- just quietly worse results.

# api_key is deliberately NOT passed. Chroma reads OPENAI_API_KEY from the
# environment, and a hand-passed key triggers a DeprecationWarning because it
# cannot be saved alongside the collection.

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)

# Prove it works and confirm the size before we depend on it.
sample = embedding_fn(["[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator"])
print("model  :", embedding_fn.model_name)
print("numbers:", len(sample[0]))
print("first 5:", [round(n, 4) for n in sample[0][:5]])

model  : text-embedding-3-small
numbers: 1536
first 5: [np.float32(-0.0306), np.float32(0.017), np.float32(-0.0597), np.float32(-0.0281), np.float32(0.0238)]


In [9]:
# A collection is one named group of documents, like a table.
# It holds the text, the embeddings, the metadata, and a record of which
# embedding function was used.

# get_or_create_collection, not create_collection: this cell gets re-run a lot,
# and create_collection raises once the name exists. That is the same trap
# behind bug B1 -- see BUGS.md.
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

print("collection:", collection.name)
print("documents :", collection.count())

collection: udaplay
documents : 15


### Add documents

In [10]:
# Read every game file, build its sentence, and load the lot into the collection.
data_dir = "games"

ids, documents, metadatas = [], [], []

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    with open(os.path.join(data_dir, file_name), "r", encoding="utf-8") as f:
        game = json.load(f)

    # The sentence that gets embedded. Platform, name, year, genre and publisher
    # are pulled IN deliberately -- anything left out cannot be found by a
    # meaning-based search, because metadata is never embedded.
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"{game['Genre']}, published by {game['Publisher']}. {game['Description']}"
    )

    ids.append(os.path.splitext(file_name)[0])   # "001.json" -> "001"
    documents.append(content)
    metadatas.append(game)                       # full JSON kept for exact lookups

# One call with all 15, not 15 separate calls. The embedding function takes a
# list so it can do them in a single request.
#
# upsert, not add: add() silently ignores an id that already exists, so editing
# the sentence above and re-running would leave the OLD text in the database
# with no warning. upsert replaces it.
collection.upsert(ids=ids, documents=documents, metadatas=metadatas)

print("documents in collection:", collection.count())
print()
print("example of what got stored:")
print(" ", documents[0])

documents in collection: 15

example of what got stored:
  [PlayStation 1] Gran Turismo (1997) - Racing, published by Sony Computer Entertainment. A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.


In [ ]:
# Persistence check.
#
# Strongest version of this test: restart the kernel, then run only the import
# cell, the load_dotenv cell, and THIS cell -- skipping the ingest entirely.
# If it still prints 15, the data genuinely came off disk and not from memory.

# A fresh client handle pointed at the same folder on disk.
check_client = chromadb.PersistentClient(path="chromadb")

# get_collection, not get_or_create_collection. This one RAISES if the
# collection is missing, so a number below proves it was already there.
check = check_client.get_collection("udaplay")

print("documents on disk:", check.count())

# Read one back by id, to confirm the metadata survived next to the text.
row = check.get(ids=["009"])
print("id 009 name     :", row["metadatas"][0]["Name"])
print("id 009 stored as:", row["documents"][0][:80], "...")